## Practice 15 - Dimensionality Reduction
In this lab we will use Dimensionality Reduction methods for classification. \
Based on Chapter 8 from Aurelien Geron's book, Hands-on Machine Learning with Scikit-Learn Keras & Tensorflow.\
Original code examples from book in github [here](https://github.com/ageron/handson-ml2)

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/dtrad/geoml_course/blob/master/Practice15DimRed.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)

%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt

### Exercise 1: 
Using the data set in 3 dimensions generated by the code below:\
a) Take its SVD, check you can reconstruct the original data (using np).\
b) Make zero one of the eigenvalues and do the same reconstruction. Plot in 3D \
c) Do a projection onto the principal axes using all dimensions. Use numpy first then sklearn.\
d) Do a projection onto the two principal axes only. Plot in 2D.\
e) Do a reconstruction in 3D from the 2D projection. Plot in 3D.

In [ ]:
from mpl_toolkits import mplot3d

The data is generated in a way that only 2 dimensions are independent and the 3rd is just a linear combination of the others plus noise.

In [ ]:
np.random.seed(4)
m = 60
w1, w2 = 0.1, 0.3
noise = 0.1
angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(X[:,0],X[:,1], X[:,2],c=X[:,2], cmap='Greens');
plt.title('Input data')

We can use numpy.linalg.svd:\
${\bf U,S,V}=svd(X)$ implies that $\bf X= U S V^T$\
notice typically svd will give you $\bf V$ (singular vectors along columns), but numpy gives you $\bf V^T$

In [ ]:
XC = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(XC)
c1 = Vt.T[:, 0]
c2 = Vt.T[:, 1]
print("V=",Vt.T)
print(c1)

In [ ]:
print(U.shape)
print(s.shape)
print(Vt.shape)

Show that U and V are orthonormal matrices

In [ ]:
print(Vt.dot(Vt.T))
UUT=(U.T.dot(U))
udiag=[UUT[i,i] for i in range(60)]
print(min(udiag), max(udiag))

Notice the "@" for matrix multiplication instead of "*". Both forms are equivalent (the @ is only python 3.X)

In [ ]:
error=(XC.dot(Vt.T)-XC@Vt.T)
print(error.shape, error.min(), error.max())

Now let us reconstruct X by multiplying the SVD components.\
s is a vector, we need a matrix size (3,3)

In [ ]:
import scipy.linalg as la
S=la.diagsvd(s,*X.shape)
XR=U@S@Vt
print("reconstruction error between %1.1e and %1.1e"%((XR - XC).min(), (XR-XC).max()))

In [ ]:
fig= plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(XR[:,0],XR[:,1], XR[:,2],c=XR[:,2], cmap='Greens');
plt.title('Reconstructed via $U * S * V^T $')

In [ ]:
# now let eliminate one of the 3 dimensions by making the smaller singular value equal to 0
print('Original singular value vector',s)
ss=np.copy(s)
ss[2]=0;
print('New singular values',ss)
SS=la.diagsvd(ss,*X.shape)
XR2=U@SS@Vt
print("reconstruction error between %1.1e and %1.1e"%((XR2 - XC).min(), (XR2-XC).max()))
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(XR2[:,0],XR2[:,1], XR2[:,2],c=XR2[:,2], cmap='Greens');
plt.title('Reconstructed via $U * S * V^T[:,2] $')
plt.show()

We see now the points are scattered along a plane (2D) embeded in 3D.

In [ ]:
#another way with numpy (without using scipy)
XC = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(XC)
m, n = X.shape
S = np.zeros(XC.shape)
S[:n, :n] = np.diag(s)
#S[2,2]=0 # uncomment to reduce dimension to 2D.
XR = U.dot(S).dot(Vt)
print("reconstruction error between %1.1e and %1.1e"%((XR - XC).min(), (XR-XC).max()))
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(XR[:,0],XR[:,1], XR[:,2],c=XR[:,2], cmap='Greens');
plt.title('Reconstructed via $U * S * V^T $')

In the previous cases we reconstructed the data by using their component ${\bf U \times S \times V^T}$.\
We can also think of V as an orthogonal transform. \
Any data multiplied by $\bf V^T$ gets transformed (similar to a forward Fourier transform), \
A transform multiplied by $\bf V$ is reconstructed (similar to an inverse Fourier transform).\
Since $\bf V^T V = I$ there is no lost. 

In [ ]:
## SVD as a transform
XC = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(XC)
XR=XC.dot(Vt.T)
XRI=XR.dot(Vt)
print("reconstruction error between %1.1e and %1.1e"%((XRI - XC).min(), (XRI-XC).max()))

Since we can map a transformed data back to its original space by multiplying by $\bf V$ \
we see the columns of $V$ are the basis functions of the transformed space.\
This is each vector in the data set is projected along the V vectors that form the basis functions of 3D.\
### Remember:
* Each column of U is a basis function of the data space.\
* Each column of V is a basis function of the model space.\
* Each element of S is the strength of the mapping between data and model.\

By eliminating one component of S or one vector of V we are reducing dimensionality.\
(when multiplying $C = A * B^T$ using matrix product, each column of C is the projection (dot product)\
of one column of A with a column of B)

In [ ]:
print(Vt.T[:,:3])

Just to clarify, you can use either @ or .dot when multiplying matrices

In [ ]:
# let us do a projection on 3D
W2 = Vt.T[:, :3] # this is just V
X3D = XC.dot(W2)
print(X3D.shape)
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(X3D[:,0],X3D[:,1], X3D[:,2],c=X3D[:,2], cmap='Greens');
plt.title("$ X_C * V $")
plt.show()

### Using sklearn PCA
Using scikit-learn is of course easier since the this code is already implemented into the PCA class\
using the familiar interface "fit_transform()" that we normally use and inverse_transform (instead of predict)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 3)
X3D = pca.fit_transform(X)

fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(X[:,0],X[:,1], X[:,2],c=X[:,2], cmap='Greens');plt.title('ORIGINAL')
plt.show()

fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(X3D[:,0],X3D[:,1], X3D[:,2],c=X3D[:,2], cmap='Greens');plt.title('TRANSFORMED')
plt.show()

X3DI = pca.inverse_transform(X3D)
ax = plt.axes(projection='3d')
ax.scatter3D(X3DI[:,0],X3DI[:,1], X3DI[:,2],c=X3DI[:,2], cmap='Greens');plt.title('RECONSTRUCTED')
plt.show()


We can see the projection in 2D by projecting the data onto the two main vectors of V.\
(using the dot product, column by column)

In [ ]:
W2 = Vt.T[:, :2] # first two columns of V
print(W2)
X2D = XC.dot(W2)
print(X2D.shape)
fig = plt.figure()
plt.plot(X2D[:,0],X[:,1],'b.'),plt.title("projected to 2D using V")
plt.show()

In [ ]:
X3D_inv = X2D.dot(Vt[:2, :]) # multiply times V^T (notice that the 3rd dimension was already wiped out)
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(X3D_inv[:,0],X3D_inv[:,1], X3D_inv[:,2],c=X3D_inv[:,2], cmap='Greens');
plt.title("reconstructed in 3D after removing last column of V")
plt.show()

In [ ]:
pca = PCA(n_components = 2)
X2D = pca.fit_transform(X)
print(X2D.shape)
fig = plt.figure()
plt.plot(X2D[:,0],X[:,1],'b.')
plt.title('projected to 2D using pca')
plt.show()

In [ ]:
X3D_inv = pca.inverse_transform(X2D)
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter3D(X3D_inv[:,0],X3D_inv[:,1], X3D_inv[:,2],c=X3D_inv[:,2], cmap='Greens');
plt.title('reconstructed to 2D with 2 components of pca')
plt.show()

### Exercise 2: Manifold Learning
Use the Swiss roll data generated below:\
a) Remove one of the dimensions by squeezing it.\
b) Remove one of the dimensions by using the values as parameters.\
c) Apply dimensionality reduction to 2 components using PCA\
d) Apply dimensionality reduction to 2 components using linear PCA\
e) Apply dimensionality reduction to 2 components using Kernel PCA


In [ ]:
# returns 
# X : array of shape [n_samples, 3], the points.
# t : array of shape [n_samples], the univariate position of the samples along of the main dimension
from sklearn.datasets import make_swiss_roll
X, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
print(X.shape,t.shape)
plt.subplot(311)
plt.plot(t,X[:,0],'.')
plt.subplot(312)
plt.plot(t,X[:,1],'.')
plt.subplot(313)
plt.plot(t,X[:,2],'.')

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = plt.axes(projection='3d')
ax.scatter3D(X[:,0],X[:,1], X[:,2],c=t, cmap=plt.cm.hot);
ax.view_init(10, -70)
plt.title('Swiss Roll data set')
plt.show()

Squeezing one dimension is easy (similar to stacking seismic data). We just plot 3D data in 2D.\
Spreading along a parameter is also easy if we have the parameter. \
Notice that is only good along the main dimension for which t is designed for.



In [ ]:
fig = plt.figure(figsize=(10,5))
plt.subplot(121)
plt.scatter(X[:,0],X[:,1],c=t, cmap=plt.cm.hot);
plt.title("a)squeezing 3rd dimension")
plt.subplot(122)
plt.scatter(t,X[:,1],c=t, cmap=plt.cm.hot);
plt.title("b) parameterized plot along values")
plt.show()

Now, we will use PCA. We calculate $\bf V$ (fit) and project (transform) in one step with fit_transform.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 2)
X2D = pca.fit_transform(X)
print(X2D.shape)
fig = plt.figure()
ax = plt.axes(projection='3d')
#ax.scatter(X2D[:,0],X2D[:,1],X2D[:,2],c=t, cmap=plt.cm.hot);
ax.scatter(X2D[:,0],X2D[:,1],c=t, cmap=plt.cm.hot);
plt.show()

In [ ]:
fig = plt.figure()
plt.scatter(X2D[:,0],X2D[:,1],c=t, cmap=plt.cm.hot)
plt.title("c) PCA")

In [ ]:
X2DR = pca.inverse_transform(X2D)
fig = plt.figure()
ax = plt.axes(projection='3d')
#ax.scatter(X2D[:,0],X2D[:,1],X2D[:,2],c=t, cmap=plt.cm.hot);
ax.scatter(X2DR[:,0],X2DR[:,1],c=t, cmap=plt.cm.hot);
plt.title('recovered after eliminating one dimension')
plt.show()

In [ ]:
from sklearn.decomposition import KernelPCA
lin_pca = KernelPCA(n_components = 2, kernel="linear", fit_inverse_transform=True)
X2Db = lin_pca.fit_transform(X)
fig = plt.figure()
plt.scatter(X2Db[:,0],X2Db[:,1],c=t, cmap=plt.cm.hot)
plt.title("d) linear PCA (transform)")

In [ ]:
X2DbI = lin_pca.inverse_transform(X2Db)
fig = plt.figure()
ax = plt.axes(projection='3d')
plt.scatter(X2DbI[:,0],X2DbI[:,1],X2DbI[:,2],c=t, cmap=plt.cm.hot)
plt.title("d) linear PCA (recovered)")

In [ ]:
fig = plt.figure()
plt.scatter(X2DbI[:,0],X2DbI[:,1],c=t, cmap=plt.cm.hot)
plt.title("d) linear PCA 2D (recovered)")

In [ ]:
from sklearn.decomposition import KernelPCA
rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.004,fit_inverse_transform=True)
X2Dc = rbf_pca.fit_transform(X)
fig = plt.figure()
plt.scatter(X2Dc[:,0],X2Dc[:,1],c=t, cmap=plt.cm.hot)
plt.title("e) Kernel PCA")

In [ ]:
X2DcI = rbf_pca.inverse_transform(X2Dc)
fig = plt.figure()
ax = plt.axes(projection='3d')
plt.scatter(X2DcI[:,0],X2DcI[:,1],X2DcI[:,2],c=t, cmap=plt.cm.hot)
plt.title("f) Kernel PCA (recovered)")

In [ ]:
fig = plt.figure()
plt.scatter(X2DcI[:,0],X2DcI[:,1],c=t, cmap=plt.cm.hot)
plt.title("g) Kernel PCA 2D (recovered)")

Now let us apply the LLE method instead of PCA.

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_reduced = lle.fit_transform(X)
plt.title("Unrolled swiss roll using LLE", fontsize=14)
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=t, cmap=plt.cm.hot)
plt.xlabel("$z_1$")
plt.ylabel("$z_2$")

### Exercise 3
Apply compression to the MNIST dataset and compare with original data.\
Use the parameter "explained_variance_ratio" to define the amount of compression.\
This parameter tells us how much of the original variance is contained on each dimension (all dimensions sum to one).\
To define the optimal number of dimensions we can sum the percentage of variances along all dimensions before the cutoff.\
Try different percentages and see how much information you loss in each case.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1)
mnist.target = mnist.target.astype(np.uint8)

In [ ]:
from sklearn.model_selection import train_test_split
X = mnist["data"]
y = mnist["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [ ]:
pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1
print('number of components necessary is ', d)
print('which is only %1.2f percent'%(d/X_train.shape[1]))
print('in dimensions but 95% in variance')

Because most variance is explained in the first dimensions, we get small lost in information but large compression.

In [ ]:
plt.plot(cumsum)
plt.xlabel("Dimensions")
plt.ylabel("Explained Variance")
plt.plot(d, 0.95, "ko")

Now that we know the optimal number of dimensions we can recover with PCA as before (fit_transform + inverse_transform)

In [ ]:
pca = PCA(n_components = 154)
X_reduced = pca.fit_transform(X_train)
X_recovered = pca.inverse_transform(X_reduced)

In [ ]:
def plot_digits(instances, images_per_row=5, **options):
    size = 28
    images_per_row = min(len(instances), images_per_row)
    images = [instance.reshape(size,size) for instance in instances]
    n_rows = (len(instances) - 1) // images_per_row + 1
    row_images = []
    n_empty = n_rows * images_per_row - len(instances)
    images.append(np.zeros((size, size * n_empty)))
    for row in range(n_rows):
        rimages = images[row * images_per_row : (row + 1) * images_per_row]
        row_images.append(np.concatenate(rimages, axis=1))
    image = np.concatenate(row_images, axis=0)
    plt.imshow(image, cmap = mpl.cm.binary, **options)
    plt.axis("off")

In [ ]:
plt.figure(figsize=(7, 4))
plt.subplot(121)
plot_digits(X_train[::2100])
plt.title("Original", fontsize=16)
plt.subplot(122)
plot_digits(X_recovered[::2100])
plt.title("Compressed", fontsize=16)

Actually we can do this directly in one pass by passing to PCA parameter "n_components" a number less than 1 (explained variance)

In [ ]:
pca = PCA(n_components = 154)
X_reduced = pca.fit_transform(X_train)
X_recovered = pca.inverse_transform(X_reduced)
plt.figure(figsize=(7, 4))
plt.subplot(121)
plot_digits(X_train[::2100])
plt.title("Original", fontsize=16)
plt.subplot(122)
plot_digits(X_recovered[::2100])
plt.title("Compressed", fontsize=16)